In [1]:
import pandas as pd
file_path=".../cambridgev2.csv"
df=pd.read_csv(file_path)
#df=df.drop(["Unnamed: 0"],axis=1)
df.head()

,ID,Text,Label,LLAMA_8B_normal_R1,LLAMA_8B_normal_R2,LLAMA_8B_normal_R3,LLAMA_8B_normal_R4,LLAMA_8B_normal_R5,LLAMA_8B_3q_R1,LLAMA_8B_3q_R2,...,AYA_32B_role_R3,AYA_32B_role_R4,AYA_32B_role_R5,AYA_32B_own_R1,AYA_32B_own_R2,AYA_32B_own_R3,AYA_32B_own_R4,AYA_32B_own_R5,FKGL,ARI
0,152,"Blacksmiths\n\nTHROUGHOUT the ages, iron has e...",4,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,9.0,12.3
1,236,The two sisters kept Lily's driving a secret f...,5,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,5.8,7.5
2,28,"Family Business\n\n'Look here, it's no good!' ...",5,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,3.0,2.6,2.8,2.8,2.4,3.0,5.9,6.5
3,137,Where I get my energy\n\nEmma Marsden asked si...,3,2.4,2.2,2.4,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,5.7,6.4
4,210,Photography\n\nWhen a photographer takes a pho...,5,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,10.3,10.9


In [3]:
df.columns[:30]

Index(['ID', 'Last Changed', 'Author', 'Title', 'Anthology', 'URL', 'Source',
       'Pub Year', 'Category', 'Location', 'License', 'MPAA\nMax',
       'MPAA \n#Max', 'MPAA\n#Avg', 'Excerpt', 'Google\nWC', 'Joon\nWC v1',
       'British WC', 'British Words', 'Sentence\nCount v1',
       'Sentence\nCount v2', 'Paragraphs', 'BT Easiness', 'BT s.e.',
       'Flesch-Reading-Ease', 'Flesch-Kincaid-Grade-Level',
       'Automated Readability Index', 'SMOG Readability',
       'New Dale-Chall Readability Formula', 'CAREC'],
      dtype='object')

In [2]:
import torch
from transformers import BertTokenizer, BertForMaskedLM
import pandas as pd
import nltk
import torch.nn as nn
import numpy as np
from sklearn.metrics import  log_loss
import math
import argparse

def calculate_score(sent_scores):
    perp_l = sorted(sent_scores, key=lambda x: x[0])
    words = [word for score, word in perp_l]
    scores = [score for score, word in perp_l]

    unk_weight = 2
    perp_l = [math.sqrt(i + 1) * prob if not words[i].startswith('##') else math.sqrt(i + 1) * prob * unk_weight for i, prob in enumerate(scores)]
    return sum(perp_l)/len(words)


In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
softmax = nn.Softmax(dim=-1)
criterion = nn.CrossEntropyLoss()
model = BertForMaskedLM.from_pretrained('bert-base-uncased')
model.cuda()
model.eval()

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'c

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

In [4]:
rsrs=[]
document_counter = 0
for index, row in df.iterrows():
        document_counter += 1
        document = row['Text']
        all_scores = []
        total_loss = 0
        sequence_length = 0
        for sent in nltk.sent_tokenize(document):
            tokenized_text = tokenizer.tokenize(sent)
            length = len(tokenized_text)
            if length > 200:
                tokenized_text = tokenized_text[:200]
                length = len(tokenized_text)
            sequence_length += length
            sent_scores = []
            for i in range(length):
                masked_index = i
                tokenized_text = tokenizer.tokenize(sent)
                if len(tokenized_text) > 200:
                    tokenized_text = tokenized_text[:200]
                true_word = tokenized_text[i]
                true_idx = tokenizer.convert_tokens_to_ids(tokenized_text)[masked_index]
                tokenized_text[masked_index] = '[MASK]'

                indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
                segments_ids = [0] * length

                tokens_tensor = torch.tensor([indexed_tokens]).cuda()
                segments_tensors = torch.tensor([segments_ids]).cuda()

                predictions = model(tokens_tensor, segments_tensors)[0]
                #loss = criterion(predictions[0, masked_index].unsqueeze(0), torch.tensor([true_idx]).cuda())
                #total_loss += loss.item()


                predictions = softmax(predictions[0, masked_index])

                y_true = np.zeros(predictions.size(0))
                y_true[true_idx] = 1.0
                nextWordLogLoss = log_loss(y_true, predictions.detach().cpu().numpy())
                sent_scores.append((nextWordLogLoss, true_word))
            score = calculate_score(sent_scores)
            all_scores.append(score)

        final_score = sum(all_scores)/len(all_scores)
        rsrs.append(final_score)

        #test_loss = total_loss / sequence_length
        #perplexity = math.exp(test_loss)

       # results.append([document, document_class, final_score, perplexity])
       # print('=' * 89)
      #  print('| Document name {} of class {} | test score {:8.8f} | length {:8.2f} | perplexity {:8.8f}'.format(document_name, document_class, final_score, len(document.split()), perplexity))
      #  print('=' * 89)
    #headers = results.pop(0)
    #df_results = pd.DataFrame(results, columns=headers)
    #df_results.to_csv(results_path, encoding='utf8', sep='\t')

In [10]:
df['RSRS_BERT']=rsrs

In [11]:
df.head()

,ID,Text,Label,LLAMA_8B_normal_R1,LLAMA_8B_normal_R2,LLAMA_8B_normal_R3,LLAMA_8B_normal_R4,LLAMA_8B_normal_R5,LLAMA_8B_3q_R1,LLAMA_8B_3q_R2,...,AYA_32B_role_R4,AYA_32B_role_R5,AYA_32B_own_R1,AYA_32B_own_R2,AYA_32B_own_R3,AYA_32B_own_R4,AYA_32B_own_R5,FKGL,ARI,RSRS_BERT
0,152,"Blacksmiths\n\nTHROUGHOUT the ages, iron has e...",4,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,9.0,12.3,0.001323
1,236,The two sisters kept Lily's driving a secret f...,5,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,5.8,7.5,0.001080
2,28,"Family Business\n\n'Look here, it's no good!' ...",5,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,2.6,2.8,2.8,2.4,3.0,5.9,6.5,0.000909
3,137,Where I get my energy\n\nEmma Marsden asked si...,3,2.4,2.2,2.4,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,5.7,6.4,0.001021
4,210,Photography\n\nWhen a photographer takes a pho...,5,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,10.3,10.9,0.001192


In [12]:
df.to_csv(".../cambridgev2.csv")

## XLMR

In [1]:
import pandas as pd
file_path=".../vikidia_frv2.csv"
df=pd.read_csv(file_path)
#df=df.drop(["Unnamed: 0"],axis=1)
df.head()

,check,File_Number,Language,Simplicity,Text,LLAMA_8B_normal_R1,LLAMA_8B_normal_R2,LLAMA_8B_normal_R3,LLAMA_8B_normal_R4,LLAMA_8B_normal_R5,...,LLAMA_70B_role_R4,LLAMA_70B_role_R5,Mixtral_normal_R1,Mixtral_role_R1,Falcon_7B_normal_R1,Falcon_7B_role_R1,FKGL,ARI,FRE,RSRS_BERT
0,Vikidia-EnFr-Parsed/frsimple/4558.html.txt,4558,fr,simple,Philippe de Belgique (Filip van België en néer...,3.4,2.6,3.2,3.4,3.8,...,4.8,4.8,4.0,3.0,4.0,4.0,5.9,9.2,87.92,0.001572
1,Vikidia-EnFr-Parsed/fr/444.html.txt,444,fr,not simple,\n\nPearl Harbor (littéralement le « port des ...,6.2,6.0,6.4,6.6,6.0,...,8.0,7.4,6.0,6.0,6.0,6.0,10.2,15.4,65.18,0.002001
2,Vikidia-EnFr-Parsed/fr/4232.html.txt,4232,fr,not simple,\n\nseptembre novembre\n\nmodifier\n\nOctobre ...,6.6,7.0,6.8,6.4,7.0,...,7.8,7.8,5.0,2.0,7.0,7.0,11.5,17.2,69.09,0.002019
3,Vikidia-EnFr-Parsed/frsimple/2944.html.txt,2944,fr,simple,Le mois de juillet est le septième mois dans l...,5.2,4.6,4.8,4.4,4.8,...,4.4,4.8,5.0,3.0,4.0,4.0,5.9,8.5,83.61,0.001436
4,Vikidia-EnFr-Parsed/frsimple/1198.html.txt,1198,fr,simple,Le cobra est un serpent venimeux à cou dilatab...,4.2,3.6,3.8,4.0,4.0,...,4.2,4.4,5.0,3.0,4.0,4.0,5.0,9.4,82.9,0.001550


In [2]:
import torch
from transformers import BertTokenizer, BertForMaskedLM
import pandas as pd
import nltk
import torch.nn as nn
import numpy as np
from sklearn.metrics import  log_loss
import math
import argparse

def calculate_score(sent_scores):
    perp_l = sorted(sent_scores, key=lambda x: x[0])
    words = [word for score, word in perp_l]
    scores = [score for score, word in perp_l]

    unk_weight = 2
    perp_l = [math.sqrt(i + 1) * prob if not words[i].startswith('##') else math.sqrt(i + 1) * prob * unk_weight for i, prob in enumerate(scores)]
    return sum(perp_l)/len(words)

In [3]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModelForMaskedLM.from_pretrained("xlm-roberta-base")
softmax = nn.Softmax(dim=-1)
criterion = nn.CrossEntropyLoss()
model.cuda()
model.eval()

Some weights of the model checkpoint at xlm-roberta-base were not used when initializing XLMRobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


XLMRobertaForMaskedLM(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True

In [ ]:
rsrs=[]
document_counter = 0
for index, row in df.iterrows():
        document_counter += 1
        document = row['Text']
        all_scores = []
        total_loss = 0
        sequence_length = 0
        try:
            for sent in nltk.sent_tokenize(document):
                tokenized_text = tokenizer.tokenize(sent)
                length = len(tokenized_text)
                if length > 200:
                    tokenized_text = tokenized_text[:200]
                    length = len(tokenized_text)
                sequence_length += length
                sent_scores = []
                for i in range(length):
                    masked_index = i
                    tokenized_text = tokenizer.tokenize(sent)
                    if len(tokenized_text) > 200:
                        tokenized_text = tokenized_text[:200]
                    true_word = tokenized_text[i]
                    true_idx = tokenizer.convert_tokens_to_ids(tokenized_text)[masked_index]
                    tokenized_text[masked_index] = '[MASK]'
    
                    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
                    segments_ids = [0] * length
    
                    tokens_tensor = torch.tensor([indexed_tokens]).cuda()
                    segments_tensors = torch.tensor([segments_ids]).cuda()
    
                    predictions = model(tokens_tensor, segments_tensors)[0]
                    #loss = criterion(predictions[0, masked_index].unsqueeze(0), torch.tensor([true_idx]).cuda())
                    #total_loss += loss.item()
    
    
                    predictions = softmax(predictions[0, masked_index])
    
                    y_true = np.zeros(predictions.size(0))
                    y_true[true_idx] = 1.0
                    nextWordLogLoss = log_loss(y_true, predictions.detach().cpu().numpy())
                    sent_scores.append((nextWordLogLoss, true_word))
                score = calculate_score(sent_scores)
                all_scores.append(score)
    
            final_score = sum(all_scores)/len(all_scores)
            rsrs.append(final_score)
        except TypeError:
            rsrs.append('na')

Token indices sequence length is longer than the specified maximum sequence length for this model (2746 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
df['RSRS_XLMR']=rsrs

In [ ]:
df.to_csv(".../vikidia_frv2.csv")

## mBERT

In [1]:
import pandas as pd
file_path=".../cambridge.csv"
df=pd.read_csv(file_path)
#df=df.drop(["Unnamed: 0"],axis=1)
df.head()

,ID,Text,Label,LLAMA_8B_normal_R1,LLAMA_8B_normal_R2,LLAMA_8B_normal_R3,LLAMA_8B_normal_R4,LLAMA_8B_normal_R5,LLAMA_8B_3q_R1,LLAMA_8B_3q_R2,...,AYA_32B_role_R5,AYA_32B_own_R1,AYA_32B_own_R2,AYA_32B_own_R3,AYA_32B_own_R4,AYA_32B_own_R5,FKGL,ARI,RSRS_BERT,RSRS_XLMR
0,152,"Blacksmiths\n\nTHROUGHOUT the ages, iron has e...",4,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,3.0,3.0,3.0,3.0,9.0,12.3,0.001323,0.000317
1,236,The two sisters kept Lily's driving a secret f...,5,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,3.0,3.0,3.0,3.0,5.8,7.5,0.001080,0.000266
2,28,"Family Business\n\n'Look here, it's no good!' ...",5,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,2.6,2.8,2.8,2.4,3.0,5.9,6.5,0.000909,0.000263
3,137,Where I get my energy\n\nEmma Marsden asked si...,3,2.4,2.2,2.4,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,5.7,6.4,0.001021,0.000269
4,210,Photography\n\nWhen a photographer takes a pho...,5,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,3.0,3.0,3.0,3.0,10.3,10.9,0.001192,0.000299


In [2]:
import torch
from transformers import BertTokenizer, BertForMaskedLM
import pandas as pd
import nltk
import torch.nn as nn
import numpy as np
from sklearn.metrics import  log_loss
import math
import argparse

def calculate_score(sent_scores):
    perp_l = sorted(sent_scores, key=lambda x: x[0])
    words = [word for score, word in perp_l]
    scores = [score for score, word in perp_l]

    unk_weight = 2
    perp_l = [math.sqrt(i + 1) * prob if not words[i].startswith('##') else math.sqrt(i + 1) * prob * unk_weight for i, prob in enumerate(scores)]
    return sum(perp_l)/len(words)

In [3]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-uncased')
model = AutoModelForMaskedLM.from_pretrained('bert-base-multilingual-uncased')
softmax = nn.Softmax(dim=-1)
criterion = nn.CrossEntropyLoss()
model.cuda()
model.eval()

BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of the model checkpoint at bert-base-multilingual-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dens

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(105879, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementw

In [4]:
rsrs=[]
document_counter = 0
for index, row in df.iterrows():
        document_counter += 1
        document = row['Text']
        all_scores = []
        total_loss = 0
        sequence_length = 0
        for sent in nltk.sent_tokenize(document):
            tokenized_text = tokenizer.tokenize(sent)
            length = len(tokenized_text)
            if length > 200:
                tokenized_text = tokenized_text[:200]
                length = len(tokenized_text)
            sequence_length += length
            sent_scores = []
            for i in range(length):
                masked_index = i
                tokenized_text = tokenizer.tokenize(sent)
                if len(tokenized_text) > 200:
                    tokenized_text = tokenized_text[:200]
                true_word = tokenized_text[i]
                true_idx = tokenizer.convert_tokens_to_ids(tokenized_text)[masked_index]
                tokenized_text[masked_index] = '[MASK]'

                indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
                segments_ids = [0] * length

                tokens_tensor = torch.tensor([indexed_tokens]).cuda()
                segments_tensors = torch.tensor([segments_ids]).cuda()

                predictions = model(tokens_tensor, segments_tensors)[0]
                #loss = criterion(predictions[0, masked_index].unsqueeze(0), torch.tensor([true_idx]).cuda())
                #total_loss += loss.item()


                predictions = softmax(predictions[0, masked_index])

                y_true = np.zeros(predictions.size(0))
                y_true[true_idx] = 1.0
                nextWordLogLoss = log_loss(y_true, predictions.detach().cpu().numpy())
                sent_scores.append((nextWordLogLoss, true_word))
            score = calculate_score(sent_scores)
            all_scores.append(score)

        final_score = sum(all_scores)/len(all_scores)
        rsrs.append(final_score)

In [6]:
df['RSRS_mBERT']=rsrs

In [8]:
df.to_csv(".../cambridge.csv")

In [7]:
df

,ID,Text,Label,LLAMA_8B_normal_R1,LLAMA_8B_normal_R2,LLAMA_8B_normal_R3,LLAMA_8B_normal_R4,LLAMA_8B_normal_R5,LLAMA_8B_3q_R1,LLAMA_8B_3q_R2,...,AYA_32B_own_R1,AYA_32B_own_R2,AYA_32B_own_R3,AYA_32B_own_R4,AYA_32B_own_R5,FKGL,ARI,RSRS_BERT,RSRS_XLMR,RSRS_mBERT
0,152,"Blacksmiths\n\nTHROUGHOUT the ages, iron has e...",4,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,3.0,3.0,3.0,9.0,12.3,0.001323,0.000317,0.000479
1,236,The two sisters kept Lily's driving a secret f...,5,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,3.0,3.0,3.0,5.8,7.5,0.001080,0.000266,0.000388
2,28,"Family Business\n\n'Look here, it's no good!' ...",5,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,2.6,2.8,2.8,2.4,3.0,5.9,6.5,0.000909,0.000263,0.000351
3,137,Where I get my energy\n\nEmma Marsden asked si...,3,2.4,2.2,2.4,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,5.7,6.4,0.001021,0.000269,0.000382
4,210,Photography\n\nWhen a photographer takes a pho...,5,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,3.0,3.0,3.0,10.3,10.9,0.001192,0.000299,0.000441
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,126,Trills and bills \n\nIf there's one thing guar...,3,2.6,2.4,2.6,2.6,2.2,2.2,2.6,...,2.2,2.0,2.2,2.4,2.0,5.7,6.1,0.000988,0.000269,0.000364
296,226,TIM RICE\n\nI was ushered into the young man's...,5,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,3.0,3.0,3.0,3.0,3.0,10.4,11.9,0.001215,0.000314,0.000459
297,67,The Ghan Train \n\nMark Ottaway travelled by t...,2,3.2,3.6,3.8,3.6,3.6,3.6,3.2,...,3.0,3.0,3.0,3.0,3.0,8.8,10.6,0.001168,0.000301,0.000420
298,25,Art on TV\n\nWhy is it that television so cons...,5,4.0,3.8,3.6,3.4,3.6,3.4,3.8,...,3.0,3.0,3.0,3.0,3.0,7.8,9.1,0.001067,0.000272,0.000380
